Combine the posture back to video

In [ ]:

import os
from pathlib import Path

HF_HOME = Path(r"D:\cache\huggingface")
os.environ["HF_HOME"] = str(HF_HOME)

# Import only AFTER the environment variable is set
from huggingface_hub import snapshot_download

POSE_DATASET_DIR = Path(
    snapshot_download(
        repo_id="SAMBOOM/AISTDance_dwpose",
        repo_type="dataset",
        # cache_dir=r"D:\cache\huggingface",
    )
)

print("Dataset location:")
print(POSE_DATASET_DIR)

# Inspect downloaded files
all_files = list(POSE_DATASET_DIR.rglob("*"))
npz_files = [p for p in all_files if p.is_file() and p.suffix == ".npz"]

print(f"\nTotal downloaded files: {sum(p.is_file() for p in all_files)}")
print(f"NPZ posture files: {len(npz_files)}")

print("\nFirst 20 NPZ files:")
for p in npz_files[:20]:
    print(p.relative_to(POSE_DATASET_DIR))

# Search for our target video
target = "gLH_sFM_c01_d16_mLH0_ch01"

print("\nMatching target:")
matches = [
    p for p in all_files
    if p.is_file() and target.lower() in p.name.lower()
]

for p in matches[:20]:
    print(p)


d:\School\Code\Rice\COMP_576\Project\beat2body\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Fetching 418 files: 100%|██████████| 418/418 [00:00<00:00, 2847.66it/s]

Dataset location:
D:\cache\huggingface\hub\datasets--SAMBOOM--AISTDance_dwpose\snapshots\2896cd444077c762b6eda89a5486cee64550c489

Total downloaded files: 418
NPZ posture files: 348

First 20 NPZ files:
2d\clips\dwpose\test\la_hiphop__gLH_sFM_c01_d16_mLH1_ch02__000000.npz
2d\clips\dwpose\test\la_hiphop__gLH_sFM_c01_d16_mLH1_ch02__000060.npz
2d\clips\dwpose\test\la_hiphop__gLH_sFM_c01_d16_mLH1_ch02__000120.npz
2d\clips\dwpose\test\la_hiphop__gLH_sFM_c01_d16_mLH1_ch02__000180.npz
2d\clips\dwpose\test\la_hiphop__gLH_sFM_c01_d16_mLH1_ch02__000240.npz
2d\clips\dwpose\test\la_hiphop__gLH_sFM_c01_d16_mLH1_ch02__000300.npz
2d\clips\dwpose\test\la_hiphop__gLH_sFM_c01_d16_mLH1_ch02__000360.npz
2d\clips\dwpose\test\la_hiphop__gLH_sFM_c01_d16_mLH1_ch02__000420.npz
2d\clips\dwpose\test\la_hiphop__gLH_sFM_c01_d16_mLH1_ch02__000480.npz
2d\clips\dwpose\test\la_hiphop__gLH_sFM_c01_d16_mLH1_ch02__000540.npz
2d\clips\dwpose\test\la_hiphop__gLH_sFM_c01_d16_mLH1_ch02__000600.npz
2d\clips\dwpose\test\la_hip

In [2]:
from pathlib import Path

# Video ID used by the HF pose chunks (see the `video_id` field printed in the
# inspection cell). All chunks for this clip belong to this one video.
VIDEO_NAME = "gLH_sFM_c01_d16_mLH0_ch01"

# Adjust if your raw videos live elsewhere
PROJECT_ROOT = Path(r"D:\School\Code\Rice\COMP_576\Project\beat2body")
VIDEO_DIR = PROJECT_ROOT / "2d" / "raw_video" / "la_hiphop"

# Try a few common AIST++ naming conventions
candidates = []
for pattern in (f"{VIDEO_NAME}.mp4", f"*{VIDEO_NAME}*.mp4"):
    candidates = sorted(VIDEO_DIR.glob(pattern))
    if candidates:
        break

if not candidates:
    raise FileNotFoundError(
        f"Could not find a video matching '{VIDEO_NAME}' in {VIDEO_DIR}"
    )

VIDEO = candidates[0]
print("Matching video:", VIDEO)

Matching video: D:\School\Code\Rice\COMP_576\Project\beat2body\2d\raw_video\la_hiphop\gLH_sFM_c01_d16_mLH0_ch01.mp4


In [3]:

import numpy as np
from pathlib import Path

VIDEO_NAME = "gLH_sFM_c01_d16_mLH0_ch01"

POSE_ROOT = (
    POSE_DATASET_DIR
    / "2d" / "clips" / "dwpose" / "train"
)

chunks = sorted(
    POSE_ROOT.glob(f"la_hiphop__{VIDEO_NAME}__*.npz")
)

print(f"Found {len(chunks)} chunks\n")

for path in chunks:
    d = np.load(path, allow_pickle=True)

    kps = d["keypoints"]
    valid = d["valid"]
    beats = d["beats"]

    valid_points = kps[valid]

    print("=" * 60)
    print("File:", path.name)
    print("Keypoints:", kps.shape)
    print("Valid points:", valid.sum(), "/", valid.size)

    print("Start:", d["start"].item())
    print("End:", d["end"].item())
    print("FPS:", d["fps"].item())
    print("Video ID:", d["video_id"].item())

    print("Beat count:", len(beats))
    print("Beat sample:", beats[:10])

    if len(valid_points):
        print("Valid XY minimum:", valid_points.min(axis=0))
        print("Valid XY maximum:", valid_points.max(axis=0))

    print("First-frame keypoints:")
    print(kps[0, :5])

print("\nOriginal video:")
print("Video:", VIDEO)


Found 20 chunks

File: la_hiphop__gLH_sFM_c01_d16_mLH0_ch01__000000.npz
Keypoints: (240, 17, 2)
Valid points: 4080 / 4080
Start: 0
End: 240
FPS: 30
Video ID: gLH_sFM_c01_d16_mLH0_ch01
Beat count: 22
Beat sample: [  2  13  24  36  47  58  69  80  92 103]
Valid XY minimum: [-1.2849774 -1.8277242]
Valid XY maximum: [1.201979  1.7480394]
First-frame keypoints:
[[ 0.01690171 -1.3550402 ]
 [ 0.08248901 -1.4109721 ]
 [-0.04988447 -1.4006664 ]
 [ 0.18653649 -1.3319286 ]
 [-0.1317     -1.3145719 ]]
File: la_hiphop__gLH_sFM_c01_d16_mLH0_ch01__000060.npz
Keypoints: (240, 17, 2)
Valid points: 4080 / 4080
Start: 60
End: 300
FPS: 30
Video ID: gLH_sFM_c01_d16_mLH0_ch01
Beat count: 21
Beat sample: [  9  20  32  43  54  66  77  88  99 110]
Valid XY minimum: [-1.2849774 -1.8277242]
Valid XY maximum: [1.0227034 1.7480394]
First-frame keypoints:
[[-0.44341263 -1.2582041 ]
 [-0.37489033 -1.3174756 ]
 [-0.44609097 -1.328356  ]
 [-0.17803498 -1.286157  ]
 [-0.36187825 -1.3068657 ]]
File: la_hiphop__gLH_sFM_c

In [4]:

if matches:
    test_file = matches[0]
    data = np.load(test_file, allow_pickle=True)

    print("File:", test_file)
    print("Available keys:", data.files)

    for key in data.files:
        print(f"{key}: shape={data[key].shape}, dtype={data[key].dtype}")
else:
    print("No matching posture file found.")


File: D:\cache\huggingface\hub\datasets--SAMBOOM--AISTDance_dwpose\snapshots\2896cd444077c762b6eda89a5486cee64550c489\2d\clips\dwpose\train\la_hiphop__gLH_sFM_c01_d16_mLH0_ch01__000000.npz
Available keys: ['keypoints', 'valid', 'mel', 'onset', 'beats', 'category', 'category_index', 'video_id', 'start', 'end', 'fps']
keypoints: shape=(240, 17, 2), dtype=float32
valid: shape=(240, 17), dtype=bool
mel: shape=(240, 64), dtype=float32
onset: shape=(240,), dtype=float32
beats: shape=(22,), dtype=int32
category: shape=(), dtype=<U9
category_index: shape=(), dtype=int64
video_id: shape=(), dtype=<U25
start: shape=(), dtype=int64
end: shape=(), dtype=int64
fps: shape=(), dtype=int64


In [5]:
import numpy as np

for path in chunks:
    with np.load(path, allow_pickle=True) as d:
        beats = d["beats"]
        kps = d["keypoints"]
        valid = d["valid"]

        print(
            f"{path.name.split('__')[-1]:>12} | "
            f"start={d['start'].item():4} | "
            f"end={d['end'].item():4} | "
            f"frames={len(kps):3} | "
            f"beats={len(beats):2} | "
            f"beat_range={beats.min() if len(beats) else '-'}:"
            f"{beats.max() if len(beats) else '-'} | "
            f"valid={valid.mean():.2%}"
        )

        if len(beats):
            print("    Beat indices:", beats[:8], "...", beats[-3:])

# Compare overlapping frames between the first two chunks
with np.load(chunks[0]) as a, np.load(chunks[1]) as b:
    overlap_a = a["keypoints"][60:240]
    overlap_b = b["keypoints"][:180]

    print("\nOverlap comparison:")
    print("Shape:", overlap_a.shape)
    print("Mean absolute difference:",
          np.mean(np.abs(overlap_a - overlap_b)))
    print("Maximum absolute difference:",
          np.max(np.abs(overlap_a - overlap_b)))

  000000.npz | start=   0 | end= 240 | frames=240 | beats=22 | beat_range=2:238 | valid=100.00%
    Beat indices: [ 2 13 24 36 47 58 69 80] ... [216 227 238]
  000060.npz | start=  60 | end= 300 | frames=240 | beats=21 | beat_range=9:234 | valid=100.00%
    Beat indices: [ 9 20 32 43 54 66 77 88] ... [212 223 234]
  000120.npz | start= 120 | end= 360 | frames=240 | beats=21 | beat_range=6:230 | valid=100.00%
    Beat indices: [ 6 17 28 39 50 62 73 84] ... [208 219 230]
  000180.npz | start= 180 | end= 420 | frames=240 | beats=22 | beat_range=2:238 | valid=100.00%
    Beat indices: [ 2 13 24 36 47 58 69 80] ... [216 227 238]
  000240.npz | start= 240 | end= 480 | frames=240 | beats=21 | beat_range=9:234 | valid=100.00%
    Beat indices: [ 9 20 32 43 54 65 77 88] ... [212 223 234]
  000300.npz | start= 300 | end= 540 | frames=240 | beats=21 | beat_range=5:230 | valid=100.00%
    Beat indices: [ 5 17 28 39 50 62 73 84] ... [208 219 230]
  000360.npz | start= 360 | end= 600 | frames=240 | 

In [6]:
import cv2

cap = cv2.VideoCapture(str(VIDEO))

print("Video width:", int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)))
print("Video height:", int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT)))
print("Video FPS:", cap.get(cv2.CAP_PROP_FPS))
print("Video frame count:", int(cap.get(cv2.CAP_PROP_FRAME_COUNT)))

cap.release()

Video width: 1920
Video height: 1080
Video FPS: 59.94005994005994
Video frame count: 2875


In [7]:
import numpy as np
from pathlib import Path

VIDEO_NAME = "gLH_sFM_c01_d16_mLH0_ch01"

POSE_ROOT = (
    POSE_DATASET_DIR
    / "2d" / "clips" / "dwpose" / "train"
)

chunks = sorted(
    POSE_ROOT.glob(f"la_hiphop__{VIDEO_NAME}__*.npz"),
    key=lambda p: int(p.stem.split("__")[-1])
)

# Determine total sequence length from chunk metadata
total_frames = max(
    int(np.load(path)["end"].item())
    for path in chunks
)

keypoints = np.zeros((total_frames, 17, 2), dtype=np.float32)
valid = np.zeros((total_frames, 17), dtype=bool)
assigned = np.zeros(total_frames, dtype=bool)

all_beats = []

for path in chunks:
    with np.load(path, allow_pickle=True) as data:
        start = int(data["start"].item())
        end = int(data["end"].item())

        local_kps = data["keypoints"]
        local_valid = data["valid"]

        # Verify that chunk metadata matches its actual array length
        assert end - start == len(local_kps)

        # Verify overlapping frames are identical
        overlap = assigned[start:end]

        if overlap.any():
            existing = keypoints[start:end][overlap]
            incoming = local_kps[overlap]

            assert np.allclose(existing, incoming, atol=1e-6), (
                f"Pose mismatch in overlapping chunk: {path.name}"
            )

        # Only write frames that have not already been assigned
        new_frames = ~assigned[start:end]

        keypoints[start:end][new_frames] = local_kps[new_frames]
        valid[start:end][new_frames] = local_valid[new_frames]
        assigned[start:end][new_frames] = True

        # Convert chunk-relative beat indices to global indices
        global_beats = data["beats"].astype(np.int64) + start
        all_beats.extend(global_beats.tolist())

assert assigned.all(), "Some frames are missing!"

beats = np.unique(all_beats)

print("Reconstructed sequence:")
print("Keypoints:", keypoints.shape)
print("Valid:", valid.shape)
print("Total frames:", len(keypoints))
print("Duration:", len(keypoints) / 30, "seconds")
print("Total beats:", len(beats))
print("Beat indices:", beats[:20])

Reconstructed sequence:
Keypoints: (1380, 17, 2)
Valid: (1380, 17)
Total frames: 1380
Duration: 46.0 seconds
Total beats: 123
Beat indices: [  2  13  24  36  47  58  69  80  92 103 114 126 137 148 159 170 182 193
 204 216]


In [8]:
import cv2

cap = cv2.VideoCapture(str(VIDEO))

video_fps = cap.get(cv2.CAP_PROP_FPS)
video_frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

pose_fps = 30

pose_indices = np.arange(len(keypoints))
video_indices = np.rint(
    pose_indices * video_fps / pose_fps
).astype(int)

# Keep only pose frames that have corresponding video frames
in_video = video_indices < video_frame_count

pose_indices = pose_indices[in_video]
video_indices = video_indices[in_video]

print("Video FPS:", video_fps)
print("Video frames:", video_frame_count)
print("Aligned pose frames:", len(pose_indices))
print("First frame mapping:", pose_indices[0], "->", video_indices[0])
print("Last frame mapping:", pose_indices[-1], "->", video_indices[-1])

cap.release()

Video FPS: 59.94005994005994
Video frames: 2875
Aligned pose frames: 1380
First frame mapping: 0 -> 0
Last frame mapping: 1379 -> 2755


In [9]:
print("Coordinate statistics:")
print("X min/max:", keypoints[:, :, 0].min(), keypoints[:, :, 0].max())
print("Y min/max:", keypoints[:, :, 1].min(), keypoints[:, :, 1].max())

print("Coordinate mean:", keypoints.mean(axis=(0, 1)))
print("Coordinate std:", keypoints.std(axis=(0, 1)))

print("\nFirst frame:")
for i, (x, y) in enumerate(keypoints[0]):
    print(f"Joint {i:2}: ({x: .4f}, {y: .4f})")

print("\nJoint coordinate ranges:")
for i in range(17):
    print(
        f"Joint {i:2}: "
        f"X [{keypoints[:, i, 0].min(): .3f}, "
        f"{keypoints[:, i, 0].max(): .3f}] "
        f"Y [{keypoints[:, i, 1].min(): .3f}, "
        f"{keypoints[:, i, 1].max(): .3f}]"
    )

Coordinate statistics:
X min/max: -1.2849774 1.3150125
Y min/max: -1.8478332 1.7480394
Coordinate mean: [ 0.00083772 -0.38281733]
Coordinate std: [0.3339982  0.89014244]

First frame:
Joint  0: ( 0.0169, -1.3550)
Joint  1: ( 0.0825, -1.4110)
Joint  2: (-0.0499, -1.4007)
Joint  3: ( 0.1865, -1.3319)
Joint  4: (-0.1317, -1.3146)
Joint  5: ( 0.3550, -0.9788)
Joint  6: (-0.2541, -0.9872)
Joint  7: ( 0.6876, -0.6243)
Joint  8: (-0.5867, -0.6062)
Joint  9: ( 0.3997, -0.7579)
Joint 10: (-0.2298, -0.8069)
Joint 11: ( 0.1976,  0.0076)
Joint 12: (-0.1976, -0.0076)
Joint 13: ( 0.1624,  0.6630)
Joint 14: (-0.2488,  0.6659)
Joint 15: ( 0.1198,  1.2670)
Joint 16: (-0.2012,  1.2680)

Joint coordinate ranges:
Joint  0: X [-0.541,  0.598] Y [-1.536,  0.056]
Joint  1: X [-0.497,  0.586] Y [-1.565,  0.026]
Joint  2: X [-0.543,  0.534] Y [-1.562,  0.048]
Joint  3: X [-0.304,  0.600] Y [-1.492, -0.322]
Joint  4: X [-0.449,  0.382] Y [-1.451, -0.310]
Joint  5: X [-0.198,  0.567] Y [-1.157, -0.430]
Joint  6:

In [10]:

import cv2
import numpy as np
from pathlib import Path
from tqdm import tqdm
from IPython.display import Video, display

# ============================================================
# 1. CONFIGURATION
# ============================================================

POSE_FPS = 30

# Save to the notebook's current working directory
OUTPUT_DIR = Path.cwd()
OUTPUT_PATH = OUTPUT_DIR / "dance_pose_visualization.mp4"

# Original video dimensions
cap = cv2.VideoCapture(str(VIDEO))

if not cap.isOpened():
    raise RuntimeError(f"Could not open video: {VIDEO}")

VIDEO_FPS = cap.get(cv2.CAP_PROP_FPS)
VIDEO_WIDTH = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
VIDEO_HEIGHT = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
VIDEO_FRAME_COUNT = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

# Output dimensions
DISPLAY_WIDTH = 960
DISPLAY_HEIGHT = 540

POSE_PANEL_SIZE = 540

OUTPUT_WIDTH = DISPLAY_WIDTH + POSE_PANEL_SIZE
OUTPUT_HEIGHT = DISPLAY_HEIGHT

print("Original video:", VIDEO)
print("Video FPS:", VIDEO_FPS)
print("Video dimensions:", VIDEO_WIDTH, "x", VIDEO_HEIGHT)
print("Pose frames:", len(keypoints))
print("Output:", OUTPUT_PATH)


# ============================================================
# 2. COCO 17-JOINT SKELETON CONNECTIONS
# ============================================================

SKELETON = [
    (0, 1), (0, 2),
    (1, 3), (2, 4),

    (5, 6),
    (5, 7), (7, 9),
    (6, 8), (8, 10),

    (5, 11), (6, 12),
    (11, 12),

    (11, 13), (13, 15),
    (12, 14), (14, 16)
]


# ============================================================
# 3. COORDINATE TRANSFORMATION FOR VISUALIZATION
# ============================================================

# Fixed coordinate limits based on the observed dataset range.
# This preserves consistent scale across all frames.

X_MIN, X_MAX = -1.5, 1.5
Y_MIN, Y_MAX = -2.0, 2.0

PANEL_MARGIN = 45

def pose_to_pixel(x, y):
    usable_width = POSE_PANEL_SIZE - 2 * PANEL_MARGIN
    usable_height = POSE_PANEL_SIZE - 2 * PANEL_MARGIN

    px = PANEL_MARGIN + int(
        (x - X_MIN) / (X_MAX - X_MIN) * usable_width
    )

    py = PANEL_MARGIN + int(
        (y - Y_MIN) / (Y_MAX - Y_MIN) * usable_height
    )

    return px, py


# ============================================================
# 4. DRAW POSE PANEL
# ============================================================

def draw_pose_panel(frame_keypoints, frame_valid, frame_index, is_beat):
    panel = np.zeros(
        (POSE_PANEL_SIZE, POSE_PANEL_SIZE, 3),
        dtype=np.uint8
    )

    # Background grid
    for x in np.linspace(X_MIN, X_MAX, 7):
        px, _ = pose_to_pixel(x, 0)
        cv2.line(
            panel,
            (px, PANEL_MARGIN),
            (px, POSE_PANEL_SIZE - PANEL_MARGIN),
            (35, 35, 35),
            1
        )

    for y in np.linspace(Y_MIN, Y_MAX, 9):
        _, py = pose_to_pixel(0, y)
        cv2.line(
            panel,
            (PANEL_MARGIN, py),
            (POSE_PANEL_SIZE - PANEL_MARGIN, py),
            (35, 35, 35),
            1
        )

    # Draw skeleton connections
    for joint_a, joint_b in SKELETON:
        if frame_valid[joint_a] and frame_valid[joint_b]:

            point_a = pose_to_pixel(*frame_keypoints[joint_a])
            point_b = pose_to_pixel(*frame_keypoints[joint_b])

            cv2.line(
                panel,
                point_a,
                point_b,
                (0, 220, 255),
                3,
                cv2.LINE_AA
            )

    # Draw joints
    for joint_index, (x, y) in enumerate(frame_keypoints):

        if not frame_valid[joint_index]:
            continue

        px, py = pose_to_pixel(x, y)

        cv2.circle(
            panel,
            (px, py),
            5,
            (255, 150, 0),
            -1,
            cv2.LINE_AA
        )

    # Frame information
    cv2.putText(
        panel,
        f"Pose Frame: {frame_index}",
        (15, 25),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.55,
        (255, 255, 255),
        1,
        cv2.LINE_AA
    )

    # Beat indicator
    if is_beat:
        cv2.circle(panel, (POSE_PANEL_SIZE - 30, 25), 9, (0, 0, 255), -1)
        cv2.putText(
            panel,
            "BEAT",
            (POSE_PANEL_SIZE - 90, 31),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.5,
            (0, 0, 255),
            2,
            cv2.LINE_AA
        )

    return panel


# ============================================================
# 5. CREATE VIDEO WRITER
# ============================================================

fourcc = cv2.VideoWriter_fourcc(*"mp4v")

writer = cv2.VideoWriter(
    str(OUTPUT_PATH),
    fourcc,
    POSE_FPS,
    (OUTPUT_WIDTH, OUTPUT_HEIGHT)
)

if not writer.isOpened():
    cap.release()
    raise RuntimeError("Could not initialize video writer.")


# ============================================================
# 6. ALIGN VIDEO AND POSE FRAMES  (no seeking)
# ============================================================

beat_set = set(beats.tolist())

# Map each pose frame to its corresponding video frame index
video_indices = np.rint(
    np.arange(len(keypoints)) * VIDEO_FPS / POSE_FPS
).astype(int)

# Drop pose frames that map past the end of the video
valid_mask   = video_indices < VIDEO_FRAME_COUNT
pose_indices = np.arange(len(keypoints))[valid_mask]
video_indices = video_indices[valid_mask]
frame_count  = len(pose_indices)

# Resize original video while preserving aspect ratio
scale = min(DISPLAY_WIDTH / VIDEO_WIDTH, DISPLAY_HEIGHT / VIDEO_HEIGHT)
resized_width  = int(VIDEO_WIDTH  * scale)
resized_height = int(VIDEO_HEIGHT * scale)
x_offset = (DISPLAY_WIDTH  - resized_width)  // 2
y_offset = (DISPLAY_HEIGHT - resized_height) // 2

print("\nGenerating visualization...")
print("Frames to render:", frame_count)


# ============================================================
# 7. RENDER EACH FRAME  (single sequential decode, no seek)
# ============================================================

target_idx      = 0          # index into pose_indices / video_indices
current_video_i = 0          # actual decoded frame counter

pbar = tqdm(total=frame_count)

while target_idx < frame_count:
    ret, original_frame = cap.read()
    if not ret:
        break

    if current_video_i == video_indices[target_idx]:
        pose_i = int(pose_indices[target_idx])

        resized_frame = cv2.resize(
            original_frame,
            (resized_width, resized_height),
            interpolation=cv2.INTER_AREA,   # better for downscaling
        )

        video_panel = np.zeros(
            (DISPLAY_HEIGHT, DISPLAY_WIDTH, 3), dtype=np.uint8
        )
        video_panel[
            y_offset:y_offset + resized_height,
            x_offset:x_offset + resized_width
        ] = resized_frame

        pose_panel = draw_pose_panel(
            keypoints[pose_i],
            valid[pose_i],
            pose_i,
            pose_i in beat_set,
        )

        writer.write(np.hstack([video_panel, pose_panel]))

        target_idx += 1
        pbar.update(1)

    current_video_i += 1

pbar.close()


# ============================================================
# 8. RELEASE RESOURCES
# ============================================================

cap.release()
writer.release()

print("\nVideo generation complete!")
print("Saved to:", OUTPUT_PATH)
print("Rendered duration:", frame_count / POSE_FPS, "seconds")


# ============================================================
# 9. DISPLAY VIDEO INSIDE NOTEBOOK
# ============================================================

display(
    Video(
        filename=str(OUTPUT_PATH),
        embed=False,
        width=1200
    )
)


Original video: D:\School\Code\Rice\COMP_576\Project\beat2body\2d\raw_video\la_hiphop\gLH_sFM_c01_d16_mLH0_ch01.mp4
Video FPS: 59.94005994005994
Video dimensions: 1920 x 1080
Pose frames: 1380
Output: d:\School\Code\Rice\COMP_576\Project\beat2body\2d\test_code\dance_pose_visualization.mp4

Generating visualization...
Frames to render: 1380


100%|██████████| 1380/1380 [00:13<00:00, 103.45it/s]


Video generation complete!
Saved to: d:\School\Code\Rice\COMP_576\Project\beat2body\2d\test_code\dance_pose_visualization.mp4
Rendered duration: 46.0 seconds
